# 2025 — Diffusion Language Models (Masked Diffusion, LLaDA)

**Papers:** Austin et al., *D3PM* (2021); Sahoo et al., *MDLM* (2024); Shen Nie et al., *"Large Language Diffusion Models"* (**LLaDA**, February 2025), an 8B model competitive with LLaMA-3 8B. In production: Inception Labs' *Mercury* (2025) and Google's *Gemini Diffusion* (May 2025) generate text with diffusion, many tokens per step.

Autoregressive LMs write strictly left to right, one token per forward pass. A **masked diffusion LM** instead:
- **forward process:** pick $t\sim U(0,1)$ and replace each token with `[MASK]` independently with probability $t$
- **model:** a *bidirectional* Transformer (no causal mask) predicts every masked token at once
- **loss:** cross-entropy on the masked positions, weighted by $1/t$, which is an upper bound on the negative log-likelihood:
$$\mathcal L = \mathbb E_{t, x_0}\Big[\frac1t \sum_{i:\,x_t^i=\text{[M]}} -\log p_\theta(x_0^i \mid x_t)\Big]$$
- **sampling:** start from an all-`[MASK]` sequence and, over $K$ steps, unmask the tokens the model is most confident about (LLaDA's *low-confidence remasking*)

Parallel decoding can be much faster, and the model sees **both directions**, so it can **infill**: fill a gap given text on both sides, which a left-to-right model cannot do directly.

![Diagram: Masked diffusion language model (LLaDA, Nie et al., 2025)](https://raw.githubusercontent.com/babushkai/neuralnetwork/main/assets/diagrams/73_2025_diffusion_language_model.png)

In [1]:
import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def get_device():
    """CUDA > Apple MPS > CPU. Force one with the env var NN_DEVICE=cpu|cuda|mps."""
    forced = os.environ.get("NN_DEVICE")
    if forced:
        return torch.device(forced)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

device = get_device()
torch.set_flush_denormal(True)   # tiny "denormal" floats are extremely slow on CPUs; treat them as 0
seed_everything(0)
print("PyTorch", torch.__version__, "| device:", device)

PyTorch 2.15.0 | device: mps


In [2]:
# A procedurally generated corpus (~290K characters) with a real held-out split.
_rng = random.Random(0)
_subjects = ["the model", "a small network", "the optimizer", "each layer", "the attention head", "the router", "a student", "the teacher"]
_verbs = ["learns", "predicts", "updates", "compresses", "routes", "remembers", "forgets", "copies"]
_objects = ["the next token", "a hidden state", "the gradient", "its weights", "a long context", "the key and value", "every expert", "the loss"]
_advs = ["quickly", "slowly", "carefully", "in parallel", "step by step", "without labels", "at scale", "once again"]
def _sentence():
    s = f"{_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)} {_rng.choice(_advs)}"
    if _rng.random() < 0.4: s += f" because {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    if _rng.random() < 0.2: s += f", then {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    return s[0].upper() + s[1:] + ". "
text = "".join(_sentence() for _ in range(4000))
chars = sorted(set(text)); V = len(chars); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], device=device)
split = int(0.9 * len(data)); train_data, val_data = data[:split], data[split:]
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(chars[i] for i in ids)
print(f"{len(text):,} characters, vocab {V}")
MASK = V; VM = V + 1                                                                    # extra [MASK] token id
L = 64

290,321 characters, vocab 31


In [3]:
class Denoiser(nn.Module):
    def __init__(self, d=192, n_layers=6, heads=6):
        super().__init__()
        self.tok, self.pos = nn.Embedding(VM, d), nn.Parameter(torch.randn(1, L, d))   # same scale as token embeddings: a bidirectional model needs position from the start
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0, activation="gelu", batch_first=True, norm_first=True)
        self.body = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)   # no causal mask: bidirectional
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V)
    def forward(self, x): return self.head(self.ln(self.body(self.tok(x) + self.pos)))

def get_batch(src, B=64):
    ix = torch.randint(0, len(src) - L, (B,), device=device)
    return src[ix[:, None] + torch.arange(L, device=src.device)]

def diffusion_loss(model, x0):
    t = torch.rand(len(x0), 1, device=device).clamp(min=1e-3)
    masked = torch.rand(x0.shape, device=device) < t                                     # each token masked with prob t
    xt = torch.where(masked, torch.full_like(x0, MASK), x0)
    ce = F.cross_entropy(model(xt).transpose(1, 2), x0, reduction="none")              # (B, L)
    return ((ce * masked) / t).sum() / masked.numel()

seed_everything(0)
model = Denoiser().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
steps = 2500
sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=steps, pct_start=0.05)
t0 = time.time()
for s in range(steps):
    loss = diffusion_loss(model, get_batch(train_data))
    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
    if s % 500 == 0: print(f"step {s:4d}  loss {loss.item():.3f}")
print(f"trained in {time.time() - t0:.0f}s")

step    0  loss 3.538


step  500  loss 2.514


step 1000  loss 0.999


step 1500  loss 0.732


step 2000  loss 0.545


trained in 322s


## Sampling by iterative unmasking

In [4]:
@torch.no_grad()
def sample(n_steps, x=None, temperature=0.7, show=False):
    model.eval()
    x = torch.full((1, L), MASK, device=device) if x is None else x.clone()
    n_masked = (x == MASK).sum().item()
    for k in range(n_steps):
        masked = x == MASK
        if not masked.any(): break
        probs = F.softmax(model(x) / temperature, -1)
        pred = torch.multinomial(probs[0], 1).T                                           # a guess for every position
        conf = probs.gather(-1, pred[..., None]).squeeze(-1).masked_fill(~masked, -1.0)
        n_unmask = math.ceil(n_masked * (k + 1) / n_steps) - (n_masked - masked.sum().item())
        keep = conf[0].topk(max(1, n_unmask)).indices                                    # commit the most confident guesses
        x[0, keep] = pred[0, keep]
        if show and k in (0, n_steps // 4, n_steps // 2, n_steps - 1):
            print(f"step {k + 1:3d}: " + "".join("_" if t == MASK else chars[t] for t in x[0].tolist()))
    return x

seed_everything(1)
print("Watching one sequence appear (64 tokens, 16 denoising steps):")
_ = sample(16, show=True)

Watching one sequence appear (64 tokens, 16 denoising steps):
step   1: _______________ _________________ ________________________ __ __
step   5: ______________s __e__________ _he opti_________________ght at st
step   9: ______________s __e_________. The optimizer ___ its weight at st
step  16: emayer predicts the a sloale. The optimizer prs its weight at st


In [5]:
print("Samples with different numbers of denoising steps (fewer steps = more tokens committed per forward pass):")
for n_steps in [4, 8, 16, 64]:
    seed_everything(2)
    print(f"{n_steps:3d} steps ({L / n_steps:4.1f} tokens/pass): " + decode(sample(n_steps)[0].tolist()))

Samples with different numbers of denoising steps (fewer steps = more tokens committed per forward pass):
  4 steps (16.0 tokens/pass):  A im lowle beathr routes a at  n   e enc. Eherooheluyes a step 


  8 steps ( 8.0 tokens/pass):  then the te moder loates a smaen the copieis every expert step 


 16 steps ( 4.0 tokens/pass):  the less. The teache ies the leaicts every expert step by step 


 64 steps ( 1.0 tokens/pass): . The routes er learns every expert slowly. The routes theights 


## Infilling: text on both sides
Give the model a prefix *and* a suffix and let it fill the middle. A bidirectional denoiser does this natively.

In [6]:
def infill(prefix, suffix, n_steps=16):
    gap = L - len(prefix) - len(suffix)
    x = torch.tensor([encode(prefix) + [MASK] * gap + encode(suffix)], device=device)
    out = decode(sample(n_steps, x)[0].tolist())
    return prefix + "[" + out[len(prefix):len(prefix) + gap] + "]" + suffix
for pre, suf in [("The teacher ", " because the router copies its weights. "),
                 ("A student learns ", " step by step. Each layer forgets the loss. ")]:
    seed_everything(3); print(infill(pre, suf))

The teacher [updates atep] because the router copies its weights. 
A student learns [the] step by step. Each layer forgets the loss. 


## Takeaways
- The same Transformer, trained to **denoise masked text** instead of to continue it, becomes a generator that writes many tokens per step and can fill gaps.
- At scale the trade-offs are still being worked out (quality per step, KV caching, variable-length output), but 2025 showed diffusion LMs can be competitive, very fast, and good at editing and infilling.